# VaR and ES Estimation

Value-at-Risk (VaR) and Expected Shortfall (ES) are the two primary regulatory risk measures for market risk capital under Basel III/FRTB.

$$\text{VaR}_\alpha = - \mathbb{F}^{-1} (1 - \alpha)$$

$$\text{ES}_\alpha = \mathbb{E}[L \mid L > \text{VaR}_\alpha]$$

The Fundamental Review of the Trading Book (FRTB) replaced the Basel II 99% VaR with 97.5% Expected Shortfall (ES) as the primary market risk capital measure. Unlike VaR, ES is a coherent risk measure that satisfies sub-additivity and is more sensitive to the severity of tail losses beyond the VaR threshold.

We implement and compare three estimation approaches:

| Method | Assumption | Strength |
|--------|-----------|----------|
| Historical Simulation (HS) | Non-parametric | Captures empirical fat tails |
| Parametric (Normal) | $r_t \sim \mathcal{N}(\mu, \sigma^2)$ | Simple and transparent; may underestimate tail risk under non-normal returns |
| GARCH(1,1)-filtered HS | Time-varying volatility | Captures volatility clustering |

All methods are evaluated at 99% VaR and 97.5% ES confidence levels using a 250-day rolling window, corresponding to approximately one year of market history.

In addition to full-sample estimates, regime-conditional VaR and ES are computed using the HMM classifications developed in Notebook 01. This allows us to examine whether downside risk differs systematically across Expansion, Normal, and Stress market environments.

In [3]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats
from arch import arch_model

import plotly.graph_objects as go
from plotly.subplots import make_subplots

warnings.filterwarnings('ignore')
np.random.seed(926)

ROOT         = Path().resolve().parent
OUT_RETURNS  = ROOT / 'data' / 'processed' / 'returns'
OUT_REGIMES  = ROOT / 'data' / 'processed' / 'regimes'
OUT_RISK     = ROOT / 'data' / 'processed' / 'risk_metrics'
OUT_RISK.mkdir(parents=True, exist_ok=True)

log_returns      = pd.read_csv(OUT_RETURNS / 'log_returns.csv', index_col='date', parse_dates=True)
portfolio_return = pd.read_csv(OUT_RETURNS / 'portfolio_return.csv', index_col='date', parse_dates=True).squeeze()

regime_labels    = pd.read_csv(OUT_REGIMES / 'regime_labels.csv', index_col='date', parse_dates=True)['regime']

## Historical Simulation VaR & ES

Historical Simulation (HS) is the most widely used VaR method in practice. It makes no distributional assumption — VaR is simply the empirical quantile of the return distribution over the lookback window:

$$\text{VaR}_\alpha^{HS} = -Q_{1-\alpha}(r_{t-T}, \ldots, r_{t-1})$$

$$\text{ES}_\alpha^{HS} = -\mathbb{E}[r_t \mid r_t < -\text{VaR}_\alpha^{HS}]$$

where $Q_{1-\alpha}$ denotes the $(1-\alpha)$ empirical quantile and $T = 250$ corresponds to approximately one year of trading days (Basel standard).

**Strengths**: captures fat tails and skewness without parametric assumptions.  
**Weakness**: all observations within the window receive equal weight, which means the recent volatility spikes are underweighted until old data rolls out.

In [7]:
WINDOW   = 250  # Basel standard: 1 year of trading days
CONF_VAR = 0.99  # 99% VaR
CONF_ES  = 0.975  # 97.5% ES  (FRTB standard)


def hs_var_es(returns, conf_var=CONF_VAR, conf_es=CONF_ES):
    r = np.asarray(returns)

    var_q = (1 - conf_var) * 100
    es_q = (1 - conf_es) * 100

    var_99 = -np.percentile(r, var_q)

    es_threshold = np.percentile(r, es_q)
    tail_losses = -r[r <= es_threshold]
    es_975 = tail_losses.mean() if len(tail_losses) > 0 else -es_threshold

    return var_99, es_975

# Rolling HS VaR and ES 
hs_var = pd.Series(index=portfolio_return.index, dtype=float, name='hs_var99')
hs_es = pd.Series(index=portfolio_return.index, dtype=float, name='hs_es975')

for t in range(WINDOW, len(portfolio_return)):
    window_returns = portfolio_return.iloc[t - WINDOW:t]
    var_t, es_t = hs_var_es(window_returns)
    hs_var.iloc[t] = var_t
    hs_es.iloc[t] = es_t

hs_var = hs_var.dropna()
hs_es  = hs_es.dropna()

print(f'\nFull-sample HS statistics:')
summary = pd.DataFrame({
    'HS VaR 99%' : hs_var,
    'HS ES 97.5%': hs_es,
}).describe().round(4)
print(summary.to_string())


Full-sample HS statistics:
       HS VaR 99%  HS ES 97.5%
count   5032.0000    5032.0000
mean       0.0295       0.0297
std        0.0177       0.0166
min        0.0108       0.0109
25%        0.0185       0.0189
50%        0.0248       0.0266
75%        0.0298       0.0304
max        0.0810       0.0759


Rolling Historical Simulation VaR and ES were estimated using a 250-trading-day rolling window, producing 5,032 daily risk estimates. The average 99% VaR is 2.95%, implying that under the empirical return distribution the portfolio would be expected to lose more than 2.95% on approximately one out of every hundred trading days. The corresponding average 97.5% ES is 2.97%, indicating that losses within the extreme left tail are only modestly larger than the VaR threshold on average.

A notable feature is the substantial time variation in estimated risk. The rolling 99% VaR ranges from 1.08% during tranquil market conditions to 8.10% during periods of severe stress, a more than sevenfold increase. This highlights the importance of dynamic risk measurement and suggests that a single static VaR estimate would fail to capture the evolving risk environment.

It is also worth noting that ES 97.5% is not necessarily larger than VaR 99% at every point in time. While ES must exceed VaR when both measures are evaluated at the same confidence level, their relative magnitude is not constrained when different confidence levels are used. In this sample, the maximum VaR 99% estimate (8.10%) exceeds the maximum ES 97.5% estimate (7.59%), reflecting differences in the portion of the tail captured by each measure rather than any inconsistency in the calculations.

## Parametric VaR & ES (Normal Distribution)

The parametric approach assumes portfolio returns are normally distributed :

$$r_t \sim \mathcal{N}(\mu, \sigma^2)$$

Under this assumption, portfolio losses are fully characterised by the first two moments of the return distribution. This assumption is consistent with the classical geometric Brownian motion framework, under which asset prices are lognormally distributed and continuously compounded returns are normally distributed. 

VaR and ES have closed-form expressions:

$$\text{VaR}_\alpha^{N} = -(\hat{\mu} + z_{1-\alpha} \hat{\sigma})$$

$$\text{ES}_\alpha^{N} = - (\hat{\mu} - \hat{\sigma} \cdot \frac{\phi(z_{1-\alpha})}{1 - \alpha})$$

where $z_{1-\alpha}$ is the standard normal quantile, and $\phi(\cdot)$ is the standard normal PDF.

**Strength**: closed-form, computationally efficient, easy to decompose.  
**Weakness**: normality assumption is rejected empirically (Notebook 01). Fat tails mean the normal model will systematically underestimate tail risk, particularly during stress periods.

In [8]:
def parametric_var_es(returns, conf_var=CONF_VAR, conf_es=CONF_ES):
    """
    Closed-form Normal VaR and ES.
    """
    r = np.asarray(returns)
    mu = r.mean()
    sigma = r.std(ddof=1)

    # Standard normal quantiles
    z_var = stats.norm.ppf(1 - conf_var) 
    z_es = stats.norm.ppf(1 - conf_es)

    var_99  = -(mu + z_var * sigma)
    es_975  = -mu + sigma * stats.norm.pdf(z_es) / (1 - conf_es)

    return var_99, es_975


# Rolling parametric VaR and ES 
param_var = pd.Series(index=portfolio_return.index, dtype=float, name='param_var99')
param_es  = pd.Series(index=portfolio_return.index, dtype=float, name='param_es975')

for t in range(WINDOW, len(portfolio_return)):
    window_returns = portfolio_return.iloc[t - WINDOW:t]
    var_t, es_t = parametric_var_es(window_returns)
    param_var.iloc[t] = var_t
    param_es.iloc[t] = es_t

param_var = param_var.dropna()
param_es = param_es.dropna()

print('Full-sample Parametric (Normal) statistics:')
summary = pd.DataFrame({
    'Param VaR 99%' : param_var,
    'Param ES 97.5%': param_es,
}).describe().round(4)
print(summary.to_string())

print(f'\nHS vs Parametric — mean estimates:')
print(f'  HS VaR 99%     : {hs_var.mean():.4f}')
print(f'  Param VaR 99%  : {param_var.mean():.4f}')
print(f'  Difference     : {(hs_var.mean() - param_var.mean()):.4f}  '
      f'(HS - Param)')
print(f'\n  HS ES 97.5%    : {hs_es.mean():.4f}')
print(f'  Param ES 97.5% : {param_es.mean():.4f}')
print(f'  Difference     : {(hs_es.mean() - param_es.mean()):.4f}  '
      f'(HS - Param)')

Full-sample Parametric (Normal) statistics:
       Param VaR 99%  Param ES 97.5%
count      5032.0000       5032.0000
mean          0.0250          0.0251
std           0.0131          0.0132
min           0.0094          0.0095
25%           0.0170          0.0171
50%           0.0219          0.0220
75%           0.0263          0.0264
max           0.0704          0.0708

HS vs Parametric — mean estimates:
  HS VaR 99%     : 0.0295
  Param VaR 99%  : 0.0250
  Difference     : 0.0045  (HS - Param)

  HS ES 97.5%    : 0.0297
  Param ES 97.5% : 0.0251
  Difference     : 0.0045  (HS - Param)


The rolling parametric Normal model produces lower risk estimates than Historical Simulation across both VaR and ES. The average 99% VaR is 2.50%, compared with 2.95% under Historical Simulation, while the average 97.5% ES is 2.51%, compared with 2.97% under Historical Simulation. In both cases, the Normal model underestimates average tail risk by approximately 45 basis points per day.

This difference is consistent with the non-normal return behaviour documented in Notebook 01. Since the parametric approach relies only on the rolling mean and standard deviation, it cannot fully capture empirical skewness, excess kurtosis, and extreme tail observations. As a result, the Normal model produces smoother and generally lower risk estimates, particularly when recent historical windows contain large downside returns.

The underestimation is particularly consequential during stress periods. When fat tails are most pronounced—precisely when accurate risk measurement matters most — the Normal model's reliance on mean and variance causes it to assign insufficient probability mass to extreme losses. This limitation is one of the primary motivations for adopting non-parametric and tail-sensitive risk measures, including Historical Simulation and Expected Shortfall.

Note that Parametric ES 97.5% is marginally above Parametric VaR 99% (2.51% versus 2.50%). Unlike the Historical Simulation estimates, where the relationship depends on the empirical shape of the tail, the Normal model produces a nearly fixed relationship between the two measures because both are determined entirely by the estimated mean and volatility. For a Gaussian distribution, the closed-form ES 97.5% is slightly larger than the closed-form VaR 99%, resulting in the observed ordering.

## GARCH(1,1) - Filtered Historical Simulation

The GARCH(1,1) model addresses the main weakness of both HS and the parametric approach, volatility clustering, large moves tend to be followed by large moves.

The conditional variance evolves as:

$$\sigma_t^2 = \omega + \alpha \epsilon_{t-1}^2 + \beta \sigma_{t-1}^2$$

where $\epsilon_t = r_t - \mu$ are the mean-adjusted returns. The three parameters satisfy $\omega > 0$, $\alpha, \beta \geq 0$, and $\alpha + \beta < 1$ (stationarity condition).

**Procedure (filtered HS)**:
1. Fit GARCH(1,1) to the 250-day window to obtain $\hat{\sigma}_t$
2. Compute standardised residuals $\hat{z}_t = \epsilon_t / \hat{\sigma}_t$
3. Scale residuals by the one-step-ahead forecast $\hat{\sigma}_{t+1}$
4. Apply HS quantiles to the scaled residuals

In [11]:
# GARCH(1,1)-filtered Historical Simulation
def garch_var_es(returns, conf_var=CONF_VAR, conf_es=CONF_ES):
    r = np.asarray(returns) * 100   # arch expects returns in % units

    try:
        model  = arch_model(r, vol='Garch', p=1, q=1, mean='Constant', dist='normal')
        result = model.fit(disp='off', show_warning=False)

        forecast     = result.forecast(horizon=1)
        sigma_next   = np.sqrt(forecast.variance.values[-1, 0]) / 100

        std_resid    = result.std_resid

        scaled_losses = -std_resid * sigma_next

        var_99  = np.percentile(scaled_losses, conf_var * 100)
        es_threshold = np.percentile(scaled_losses,  conf_es * 100)
        tail = scaled_losses[scaled_losses >= es_threshold]
        es_975 = tail.mean() if len(tail) > 0 else var_99

        return var_99, es_975

    except Exception:
        return hs_var_es(returns, conf_var, conf_es)


# Rolling GARCH VaR and ES
print('Fitting rolling GARCH(1,1) models')
print(f'Total windows: {len(portfolio_return) - WINDOW}')

garch_var = pd.Series(index=portfolio_return.index, dtype=float, name='garch_var99')
garch_es = pd.Series(index=portfolio_return.index,dtype=float, name='garch_es975')

for t in range(WINDOW, len(portfolio_return)):
    if t % 500 == 0:
        print(f'Progress: {t}/{len(portfolio_return)}')
    window_returns = portfolio_return.iloc[t - WINDOW:t]
    var_t, es_t = garch_var_es(window_returns)
    garch_var.iloc[t] = var_t
    garch_es.iloc[t]  = es_t

garch_var = garch_var.dropna()
garch_es = garch_es.dropna()

print(f'\nFull-sample GARCH statistics:')
summary = pd.DataFrame({
    'GARCH VaR 99%' : garch_var,
    'GARCH ES 97.5%': garch_es,
}).describe().round(4)
print(summary.to_string())

print(f'\nMethod comparison — mean estimates:')
comparison = pd.DataFrame({
    'VaR 99%'  : [hs_var.mean(), param_var.mean(), garch_var.mean()],
    'ES 97.5%' : [hs_es.mean(),  param_es.mean(),  garch_es.mean()],
}, index=['Historical Simulation', 'Parametric (Normal)', 'GARCH(1,1)-filtered'])
print(comparison.round(4).to_string())

Fitting rolling GARCH(1,1) models
Total windows: 5032
Progress: 500/5282
Progress: 1000/5282
Progress: 1500/5282
Progress: 2000/5282
Progress: 2500/5282
Progress: 3000/5282
Progress: 3500/5282
Progress: 4000/5282
Progress: 4500/5282
Progress: 5000/5282

Full-sample GARCH statistics:
       GARCH VaR 99%  GARCH ES 97.5%
count      5032.0000       5032.0000
mean          0.0266          0.0274
std           0.0193          0.0202
min           0.0089          0.0090
25%           0.0168          0.0175
50%           0.0211          0.0216
75%           0.0289          0.0299
max           0.2495          0.2656

Method comparison — mean estimates:
                       VaR 99%  ES 97.5%
Historical Simulation   0.0295    0.0297
Parametric (Normal)     0.0250    0.0251
GARCH(1,1)-filtered     0.0266    0.0274


In [12]:
print(garch_var.sort_values(ascending=False).head(10))
print(garch_es.sort_values(ascending=False).head(10))

date
2020-03-17    0.249498
2020-03-18    0.223385
2020-03-19    0.219365
2020-03-16    0.219256
2020-03-25    0.216277
2020-03-26    0.189460
2020-03-20    0.187828
2020-03-27    0.187125
2020-03-13    0.185493
2020-03-23    0.177685
Name: garch_var99, dtype: float64
date
2020-03-17    0.265579
2020-03-18    0.239842
2020-03-19    0.236392
2020-03-16    0.234644
2020-03-25    0.228681
2020-03-26    0.200379
2020-03-20    0.198780
2020-03-13    0.198243
2020-03-27    0.197947
2020-03-23    0.188045
Name: garch_es975, dtype: float64


Rolling GARCH-filtered Historical Simulation produces average risk estimates between the Historical Simulation and Parametric Normal approaches. The mean 99% VaR is 2.66%, compared with 2.95% under Historical Simulation and 2.50% under the Normal model. Similarly, the mean 97.5% ES is 2.74%, lying between the Historical Simulation estimate of 2.97% and the Parametric estimate of 2.51%.

This ordering is economically intuitive. Historical Simulation directly incorporates past extreme observations and therefore generates the highest average risk estimates. The Parametric model assumes normally distributed returns and consequently assigns insufficient probability mass to extreme losses, producing the lowest estimates. GARCH-filtered Historical Simulation sits between the two by combining empirical tail information with a time-varying volatility forecast.

A key advantage of the GARCH approach emerges during periods of market stress. While its average risk estimates remain moderate, the model responds aggressively to volatility shocks. The largest VaR and ES estimates occur during the March 2020 COVID market crisis, reaching 24.95% and 26.56%, respectively. This reflects the ability of the conditional volatility forecast to rapidly adapt to changing market conditions. The high GARCH maxima relative to the Historical Simulation maxima (8.10% VaR and 7.59% ES) reflect a fundamental difference between the two approaches. Historical Simulation is bounded by the most extreme observation contained within the rolling window, whereas GARCH can extrapolate beyond historical extremes when the one-step-ahead volatility forecast becomes exceptionally large. During March 2020, the model identified an unprecedented volatility regime and projected forward-looking risk substantially above that implied by the trailing historical sample, illustrating the responsiveness of volatility-based risk models during crisis periods.

Overall, the three methods imply materially different levels of market risk. Historical Simulation captures empirical tail behaviour, the Parametric approach provides a computationally efficient benchmark, and GARCH-filtered Historical Simulation offers a compromise between responsiveness and distributional flexibility. The next sections evaluate these models through visualization, backtesting, and regime-conditional analysis.

## Rolling VaR Comparison

By plotting all three rolling VaR estimates together with the realised portfolio return, we can visually assess model behaviour over time. Note that exceedances are days when the realised loss exceeds the VaR estimate, and a well-calibrated 99% VaR should produce exceedances approximately 1% of the time.

In [17]:
common_idx  = hs_var.index.intersection(param_var.index).intersection(garch_var.index)
port_aligned = portfolio_return.reindex(common_idx)

REGIME_COLORS = {
    'expansion': '#2ecc71',
    'normal': '#f39c12',
    'stress': '#e74c3c',
}

fig = make_subplots(
    rows=2, cols=1,
    subplot_titles=[
        'Rolling 99% VaR — Three Methods vs Realised Return',
        'VaR Exceedances (Realised Loss > VaR)',
    ],
    shared_xaxes=True,
    vertical_spacing=0.08,
    row_heights=[0.65, 0.35],
)

# VaR lines, realised return
reg_plot = regime_labels.reindex(common_idx).dropna()
current = reg_plot.iloc[0]
start = reg_plot.index[0]
for date, reg in reg_plot.iloc[1:].items():
    if reg != current:
        fig.add_vrect(
            x0=start, x1=date,
            fillcolor=REGIME_COLORS[current],
            opacity=0.10, layer='below', line_width=0,
            row=1, col=1,
        )
        start = date; current = reg
fig.add_vrect(
    x0=start, x1=reg_plot.index[-1],
    fillcolor=REGIME_COLORS[current],
    opacity=0.10, layer='below', line_width=0,
    row=1, col=1,
)

# Realised return (loss = negative return)
fig.add_trace(go.Scatter(
    x=common_idx, y=-port_aligned.values,
    mode='lines', line=dict(color='#2c3e50', width=0.8),
    name='Realised loss', opacity=0.6,
), row=1, col=1)

# VaR lines
for var_series, color, label in [
    (hs_var, '#e74c3c', 'HS VaR 99%'),
    (param_var, '#3498db', 'Parametric VaR 99%'),
    (garch_var, '#9b59b6', 'GARCH VaR 99%'),
]:
    fig.add_trace(go.Scatter(
        x=var_series.reindex(common_idx).index,
        y=var_series.reindex(common_idx).values,
        mode='lines', line=dict(color=color, width=1.2),
        name=label,
    ), row=1, col=1)

# Exceedances 
for var_series, color, label in [
    (hs_var, '#e74c3c', 'HS exceedances'),
    (param_var, '#3498db', 'Parametric exceedances'),
    (garch_var, '#9b59b6', 'GARCH exceedances'),
]:
    exceed_mask = -port_aligned > var_series.reindex(common_idx)
    exceed_dates = common_idx[exceed_mask]
    fig.add_trace(go.Scatter(
        x=exceed_dates,
        y=[-port_aligned[d] for d in exceed_dates],
        mode='markers',
        marker=dict(color=color, size=4, symbol='circle'),
        name=label,
    ), row=2, col=1)

fig.add_hline(y=0, line_dash='dash', line_color='grey', line_width=0.8, row=2, col=1)

fig.update_yaxes(title_text='Return / VaR', row=1)
fig.update_yaxes(title_text='Loss magnitude', row=2)
fig.update_layout(
    title = 'Rolling VaR Comparison — HS vs Parametric vs GARCH (2005–2025)',
    height = 650,
    hovermode = 'x unified',
    legend = dict(orientation='h', y=-0.08),
)
fig.show()

# summary 
print('Exceedance analysis (99% VaR — expected: 1% of days):')
print('-' * 55)
for var_series, label in [
    (hs_var, 'Historical Simulation'),
    (param_var, 'Parametric (Normal)  '),
    (garch_var, 'GARCH(1,1)-filtered  '),
]:
    exceed = (-port_aligned > var_series.reindex(common_idx)).sum()
    total = len(common_idx)
    fail_rate = exceed / total * 100
    print(f'{label}: {exceed:3d} exceedances in {total} days = {fail_rate:.2f}%  (expected 1.00%)')

Exceedance analysis (99% VaR — expected: 1% of days):
-------------------------------------------------------
Historical Simulation:  86 exceedances in 5032 days = 1.71%  (expected 1.00%)
Parametric (Normal)  : 133 exceedances in 5032 days = 2.64%  (expected 1.00%)
GARCH(1,1)-filtered  :  78 exceedances in 5032 days = 1.55%  (expected 1.00%)


The rolling VaR comparison shows clear differences in model responsiveness and exceedance behaviour across the three approaches. Historical Simulation produces relatively step-like VaR estimates because the empirical quantile changes only when extreme observations enter or leave the 250-day rolling window. The Parametric Normal model is smoother, but it remains systematically lower during several stress episodes, reflecting its reliance on mean and variance under a normality assumption. GARCH-filtered Historical Simulation is the most reactive of the three methods, with sharp VaR spikes during volatility shocks, especially during the March 2020 COVID crisis.

The exceedance analysis provides an initial validation of the 99% VaR forecasts. Under a correctly calibrated 99% VaR model, approximately 1% of observations should exceed the VaR threshold. Historical Simulation records 86 exceedances over 5,032 days, corresponding to a 1.71% exceedance rate. The Parametric Normal model performs worst, with 133 exceedances, or 2.64%, confirming that it materially underestimates tail risk. GARCH-filtered Historical Simulation records 78 exceedances, or 1.55%, the closest of the three to the expected 1% level.

Overall, the results suggest that incorporating volatility dynamics improves VaR responsiveness and reduces exceedance frequency relative to both standard Historical Simulation and the Normal parametric benchmark. However, all three models exceed the target 1% violation rate, indicating that formal statistical backtesting is required before drawing firm conclusions about model adequacy.

## Regime-Conditional VaR & ES

We compute VaR and ES separately for each HMM regime by subsetting the rolling estimates to days classified as Expansion, Normal, or Stress. This will reveal how risk measures vary systematically across market environments and quantifies the additional tail risk present during stress periods.

Note that we use the rolling estimates (not re-estimated within each regime) to preserve the real-time nature of the risk measures.

In [20]:
regime_aligned = regime_labels.reindex(common_idx)

results_regime = {}

for method, var_series, es_series in [
    ('Historical Simulation', hs_var,    hs_es),
    ('Parametric (Normal)', param_var, param_es),
    ('GARCH(1,1)-filtered', garch_var, garch_es),
]:
    rows = []
    for regime in ['expansion', 'normal', 'stress']:
        mask    = regime_aligned == regime
        var_reg = var_series.reindex(common_idx)[mask]
        es_reg  = es_series.reindex(common_idx)[mask]
        rows.append({
            'Regime': regime.capitalize(),
            'N days': int(mask.sum()),
            'VaR 99% mean': var_reg.mean().round(4),
            'VaR 99% max': var_reg.max().round(4),
            'ES 97.5% mean': es_reg.mean().round(4),
            'ES 97.5% max': es_reg.max().round(4),
        })
    results_regime[method] = pd.DataFrame(rows).set_index('Regime')

for method, df in results_regime.items():
    print(f'\n{method}:')
    print(df.to_string())

# Stress-to-expansion ratio
print('\nStress/Expansion VaR ratio (mean):')
print('-' * 45)
for method, df in results_regime.items():
    ratio = df.loc['Stress', 'VaR 99% mean'] / df.loc['Expansion', 'VaR 99% mean']
    print(f'{method:<30}: {ratio:.2f}x')


Historical Simulation:
           N days  VaR 99% mean  VaR 99% max  ES 97.5% mean  ES 97.5% max
Regime                                                                   
Expansion    2434        0.0220        0.081         0.0224        0.0759
Normal       1839        0.0317        0.081         0.0318        0.0759
Stress        759        0.0484        0.081         0.0475        0.0759

Parametric (Normal):
           N days  VaR 99% mean  VaR 99% max  ES 97.5% mean  ES 97.5% max
Regime                                                                   
Expansion    2434        0.0193       0.0548         0.0194        0.0551
Normal       1839        0.0269       0.0704         0.0270        0.0707
Stress        759        0.0387       0.0704         0.0389        0.0708

GARCH(1,1)-filtered:
           N days  VaR 99% mean  VaR 99% max  ES 97.5% mean  ES 97.5% max
Regime                                                                   
Expansion    2434        0.0172       0.0387

Regime-conditional VaR and ES estimates show that downside risk increases monotonically from Expansion to Normal to Stress regimes across all three methods. Under Historical Simulation, mean 99% VaR rises from 2.20% in Expansion to 4.84% in Stress. Under the Parametric Normal model, the corresponding increase is from 1.93% to 3.87%. The GARCH-filtered approach exhibits the strongest regime sensitivity, with mean VaR rising from 1.72% in Expansion to 5.69% in Stress.

The Stress-to-Expansion VaR ratio highlights these differences clearly. Historical Simulation implies a 2.20× increase in VaR, the Parametric Normal model implies a 2.01× increase, while GARCH-filtered Historical Simulation implies a 3.31× increase. This confirms that the GARCH-filtered method is the most responsive to regime shifts, particularly because it incorporates time-varying volatility forecasts.

The maximum risk estimates also differ substantially across methods. Historical Simulation reaches the same maximum VaR across regimes because old crisis observations remain inside the rolling window even after the market regime changes. In contrast, GARCH-filtered estimates are sharply concentrated in the Stress regime, with maximum VaR reaching 24.95% and maximum ES reaching 26.56%. This behaviour is consistent with a forward-looking volatility model that reacts strongly during acute market dislocations.

In [21]:
# Regime-conditional VaR box plots
REGIME_COLORS = {
    'expansion': '#2ecc71',
    'normal': '#f39c12',
    'stress': '#e74c3c',
}

methods = [
    ('Historical Simulation', hs_var, '#e74c3c'),
    ('Parametric (Normal)', param_var, '#3498db'),
    ('GARCH(1,1)-filtered', garch_var, '#9b59b6'),
]

fig = make_subplots(
    rows=1, cols=3,
    subplot_titles=[m[0] for m in methods],
    shared_yaxes=True,
    horizontal_spacing=0.05,
)

for col_idx, (method, var_series, color) in enumerate(methods, start=1):
    for regime in ['expansion', 'normal', 'stress']:
        mask = regime_aligned == regime
        var_reg = var_series.reindex(common_idx)[mask].dropna()

        fig.add_trace(
            go.Box(
                y = var_reg.values,
                name = regime.capitalize(),
                marker = dict(color=REGIME_COLORS[regime]),
                boxmean = True,
                showlegend= col_idx == 1,
            ),
            row=1, col=col_idx,
        )

fig.update_yaxes(title_text='VaR 99% (daily)', row=1, col=1)
fig.update_layout(
    title = 'Regime-Conditional VaR 99% Distribution by Method',
    height = 500,
    boxmode = 'group',
    hovermode = 'closest',
    legend = dict(orientation='h', y=-0.12),
)
fig.show()

The distributional view reinforces the regime-dependent nature of market risk. Across all three methods, the entire VaR distribution shifts upward as market conditions deteriorate from Expansion to Normal to Stress. Not only do average risk estimates increase, but the dispersion of VaR estimates also widens, indicating greater uncertainty during stressed market environments.

The difference is particularly significant for the GARCH-filtered approach. While Expansion and Normal regimes exhibit relatively compact VaR distributions, the Stress regime shows a substantial right tail with numerous extreme observations. This reflects the model's ability to rapidly amplify risk estimates when conditional volatility rises, producing substantially larger tail-risk forecasts than either Historical Simulation or the Parametric Normal model.

In contrast, Historical Simulation and Parametric VaR exhibit more compressed distributions across regimes. Although both methods recognize higher risk during Stress periods, their estimates remain bounded by either the historical sample (HS) or the normality assumption (Parametric), limiting their responsiveness during severe market dislocations.

In [23]:
# Save outputs

risk_estimates = pd.DataFrame({
    'hs_var99': hs_var,
    'hs_es975': hs_es,
    'param_var99': param_var,
    'param_es975': param_es,
    'garch_var99': garch_var,
    'garch_es975': garch_es,
}).reindex(common_idx)

risk_estimates.to_csv(OUT_RISK / 'rolling_var_es.csv')

regime_summary_rows = []
for method, var_series, es_series in [
    ('hs', hs_var, hs_es),
    ('param', param_var, param_es),
    ('garch', garch_var, garch_es),
]:
    for regime in ['expansion', 'normal', 'stress']:
        mask = regime_aligned == regime
        var_reg = var_series.reindex(common_idx)[mask]
        es_reg = es_series.reindex(common_idx)[mask]
        regime_summary_rows.append({
            'method' : method,
            'regime' : regime,
            'n_days' : int(mask.sum()),
            'var99_mean' : round(var_reg.mean(), 4),
            'var99_std' : round(var_reg.std(),  4),
            'var99_max' : round(var_reg.max(),  4),
            'es975_mean' : round(es_reg.mean(),  4),
            'es975_max' : round(es_reg.max(),   4),
        })

regime_summary = pd.DataFrame(regime_summary_rows)
regime_summary.to_csv(OUT_RISK / 'regime_var_es_summary.csv', index=False)

exceed_rows = []
for method, var_series, label in [
    ('hs', hs_var, 'Historical Simulation'),
    ('param', param_var, 'Parametric (Normal)'),
    ('garch', garch_var, 'GARCH(1,1)-filtered'),
]:
    var_aligned = var_series.reindex(common_idx)
    exceed_mask = -port_aligned > var_aligned
    n_exceed = exceed_mask.sum()
    exceed_rows.append({
        'method' : label,
        'n_exceedances'  : int(n_exceed),
        'n_days' : len(common_idx),
        'exceedance_rate': round(n_exceed / len(common_idx) * 100, 3),
        'expected_rate' : 1.0,
    })

exceed_df = pd.DataFrame(exceed_rows)
exceed_df.to_csv(OUT_RISK / 'exceedance_summary.csv', index=False)

## Key Findings

1. **All three methods confirm substantial time-variation in portfolio risk.** Rolling 99% VaR ranges from approximately 1% during calm periods to over 8% (HS) and 25% (GARCH) during acute stress, highlighting the inadequacy of static risk estimates.

2. **The Parametric Normal model systematically underestimates tail risk.** With a mean VaR 18% below Historical Simulation and the highest exceedance rate (2.64%), the normality assumption fails to capture the fat-tailed return behaviour documented in Notebook 01.

3. **GARCH(1,1)-filtered HS produces the most regime-sensitive estimates.** Its Stress-to-Expansion VaR ratio (3.31×) is the highest of the three methods, reflecting its ability to rapidly amplify risk forecasts when conditional volatility rises. The model also records the lowest exceedance rate among the three approaches (1.55%), though all models remain above the theoretical 1% target.

4. **Risk measures are strongly regime-dependent across all methods.** Mean VaR approximately doubles from Expansion to Stress under HS and Parametric, and triples under GARCH. This confirms that ignoring market regime leads to material underestimation of tail risk during stress periods.

5. **All three models exceed the theoretical 1% violation rate.** Historical Simulation records 1.71% exceedances, Parametric Normal 2.64%, and GARCH-filtered HS 1.55%. While these exceedance counts provide an initial diagnostic, formal statistical backtesting is required to determine whether the observed deviations are statistically significant.

In Notebook 03, we will perform VaR Backtesting & Model Validation including z-test, Kupiec unconditional coverage, Christoffersen conditional coverage, and PIT-based backtesting.